# Build Auxiliary Model - Package

This notebook builds the auxiliary package from a package-based dynamic model.

It takes the input package from `models/`, replaces each dynamic component with its static counterpart, adds the INIT model where there is one, removes the dynamic events, and saves the auxiliary package to `outputs/`.

In [ ]:
import logging
import os
import shutil
import sys

sys.path.append(os.path.abspath(".."))

logging.getLogger("OMPython").setLevel(logging.ERROR)

from OMPython import OMCSessionLocal

from scripts.helpers import *

# --- Configuration ---

# 1. Root model to build (qualified package name)
MODEL = "Demo_IEEE14.IEEE14DisconnectLine"

# 2. Directory containing the source package
SOURCE_PACKAGE = MODEL.split(".")[0]
MODEL_DIR = os.path.abspath(os.path.join("models", SOURCE_PACKAGE))
MODELS_PKG_PATH = os.path.join(MODEL_DIR, "package.mo")

# 3. Modelica Standard Library
OMLIB_DIR = os.path.join(os.path.expanduser("~"), ".openmodelica", "libraries")
os.environ["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = os.path.join(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")

# 4. Dynawo Modelica library
DYNAWO_PKG_PATH = os.path.abspath(
    os.path.join("..", "dynawo_library", "Dynawo", "package.mo")
)

# 5. INIT model selection for components with multiple INIT profiles (leave empty for default).
INIT_MODEL_BY_COMPONENT = {}

# 6. Slack component (leave empty to disable slack-specific handling).
SLACK_COMPONENT = "Gen1"

## Load and Validate the Model

The original package root model is loaded and checked, and the user configuration is validated against it, before any changes are made.

In [ ]:
# 1. Start OMC and load libraries
omc = OMCSessionLocal()
omc_call(omc, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(omc, "loadModel(Complex)")
omc_call(omc, "loadModel(ModelicaServices)")
omc_call(omc, f'loadFile("{DYNAWO_PKG_PATH}")')

# 2. Load the user's case and validate the configuration against it
omc_call(omc, f'loadFile("{MODELS_PKG_PATH}")')
config = check_user_configuration_package(
    omc,
    model=MODEL,
    slack_component=SLACK_COMPONENT,
    init_model_by_component=INIT_MODEL_BY_COMPONENT,
)
chain = config["model_chain"]

## Generate the Auxiliary Package

The auxiliary package is generated from a copy of the original, transforming each model in the inheritance chain, and saved to `outputs/`.

In [ ]:
OUTPUT_DIR = os.path.abspath("outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATHS = package_workflow_paths(MODEL, MODEL_DIR, OUTPUT_DIR)

AUX_PACKAGE = PATHS["aux_package"]
AUX_DIR = PATHS["aux_dir"]
AUX_ROOT_MODEL = PATHS["aux_root_model"]
AUX_PACKAGE_FILE = PATHS["aux_package_file"]
AUX_ORDER_FILE = PATHS["aux_order_file"]

AUX_NAME_MAP = auxiliary_name_map(chain, AUX_PACKAGE)

In [ ]:
package_context = collect_package_component_contexts(omc, chain)
components_by_model = package_context["components_by_model"]
patch_components_by_model = package_context["patch_components_by_model"]
global_blacklist_names = package_context["global_blacklist_names"]

# Create an empty auxiliary package in OpenModelica
send_expression(omc, f"deleteClass({AUX_PACKAGE})")
omc_call(omc, f'loadString("within ; package {AUX_PACKAGE} end {AUX_PACKAGE};")', parsed=False)

# Loop over the inheritance chain and transform each class
for model in chain:
    aux_model = AUX_NAME_MAP[model]
    aux_name = aux_model.split(".")[-1]

    print(f"Transforming {model} -> {aux_model}")
    omc_call(omc, f'copyClass({model}, "{aux_name}", {AUX_PACKAGE})')

    components = components_by_model[model]

    apply_replacements(omc, model, aux_model, components, SLACK_COMPONENT)
    delete_connections(omc, aux_model, components, global_targets=global_blacklist_names)
    delete_components(omc, aux_model, components)
    add_init_models(omc, model, aux_model, components, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)
    apply_LF_modifiers(omc, model, aux_model, components)
    add_init_equations(omc, model, aux_model, components, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)

# Write the auxiliary package to disk
if os.path.isdir(AUX_DIR):
    shutil.rmtree(AUX_DIR)
os.makedirs(AUX_DIR, exist_ok=True)

write_package_files(
    AUX_PACKAGE_FILE,
    AUX_ORDER_FILE,
    AUX_PACKAGE,
    package_class_names(chain, AUX_NAME_MAP),
)

save_auxiliary_package_classes(
    omc,
    chain,
    AUX_NAME_MAP,
    AUX_DIR,
    patch_components_by_model,
    SLACK_COMPONENT,
)

# Re-load the generated auxiliary package from disk and validate it
omc_call(omc, f"deleteClass({AUX_PACKAGE})")
omc_call(omc, f'loadFile("{AUX_PACKAGE_FILE}")')
chk = send_expression(omc, f"checkModel({AUX_ROOT_MODEL})", parsed=False)
print(chk)

print("Wrote auxiliary package:", AUX_DIR)